# M3 Vision Branch — Colab GPU Training

**Thành viên 3:** Nhánh Vision (CNN Encoders & Attention Pooling)

Notebook này chạy trên Google Colab GPU (T4) để train:
1. **Custom CNN 30 epochs** + Early Stopping
2. **ResNet-18 30 epochs** + Early Stopping

Kết quả lưu vào Google Drive → tải về đưa vào `m3_vision/results/`.

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Clone repo & cài dependency

In [ ]:
# Thay YOUR_REPO_URL bằng URL repo thật (HTTPS hoặc SSH)
# Hoặc upload project lên Colab trực tiếp
# !git clone YOUR_REPO_URL /content/Master-UIT-MedSignal
# %cd /content/Master-UIT-MedSignal

# --- HOẶC: Mount Google Drive và copy project ---
from google.colab import drive
drive.mount('/content/drive')

# Đổi đường dẫn cho đúng vị trí trên Drive của bạn
# PROJECT_ROOT = "/content/drive/MyDrive/Master-UIT-MedSignal"
# import os; os.chdir(PROJECT_ROOT)

# Cài dependency
!pip install -q scikit-learn torchvision pandas pyyaml Pillow

## 2. Upload project (nếu không clone)

Chạy cell dưới để upload file zip của project, hoặc skip nếu đã clone ở bước 1.

In [ ]:
# --- Option: Upload zip ---
# from google.colab import files
# uploaded = files.upload()  # upload Master-UIT-MedSignal.zip
# !unzip -q Master-UIT-MedSignal.zip -d /content/
# %cd /content/Master-UIT-MedSignal

import os
# Đặt PROJECT_ROOT đúng thư mục gốc
PROJECT_ROOT = os.getcwd()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
!ls configs/ src/ m3_vision/

## 3. Quick sanity check — import & 1 batch

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

import torch
from src.data import preprocess as P
from src.data.dataset import CarotidDataset, collate_fn
from src.data.splits import stratified_folds
from src.models.vision import VisionPlaqueClassifier
from src.eval.metrics import classification_metrics

cfg = P.load_config(os.path.join(PROJECT_ROOT, "configs/config.yaml"))
df = P.load_dataframe(cfg, project_root=PROJECT_ROOT)
print(f"Dataset: {len(df)} ca, Plaque=1: {df[cfg['columns']['target_plaque']].sum()}")

# Test 1 batch
from torch.utils.data import DataLoader
scaler = P.fit_scaler(P.encode_categorical(df, cfg), cfg)
ds = CarotidDataset(df, cfg, scaler, project_root=PROJECT_ROOT)
loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))
print(f"batch keys: {list(batch.keys())}")
print(f"imt_img: {batch['imt_img'].shape}")
print(f"cca_imgs: {batch['cca_imgs'].shape}")
print(f"cca_mask: {batch['cca_mask'].shape}")
print(f"plaque labels: {batch['labels']['plaque'].squeeze().tolist()}")
print("\n✅ Data pipeline OK")

## 4. Train Custom CNN — 30 epochs + Early Stopping

Expected: ~10-15 min trên T4 GPU.

In [ ]:
!python3 m3_vision/scripts/train_vision_baseline.py \
  --project-root . \
  --encoder custom_cnn \
  --epochs 30 \
  --folds 5 \
  --batch-size 16 \
  --lr 3e-4 \
  --weight-decay 1e-4 \
  --early-stop \
  --early-stop-patience 7 \
  --checkpoint-dir m3_vision/checkpoints/custom_cnn_30epoch \
  --output m3_vision/results/vision_custom_cnn_30epoch_metrics.json

In [ ]:
# Xem kết quả
import json
with open("m3_vision/results/vision_custom_cnn_30epoch_metrics.json") as f:
    result = json.load(f)

print("="*60)
print(f"Custom CNN — {result['epochs']} epochs, {result['folds']} folds")
print("="*60)
for k, v in result['summary'].items():
    print(f"  {k:15s}: {v['mean']:.4f} ± {v['std']:.4f}")
print()
print("Per-fold:")
for fr in result['fold_results']:
    m = fr['best_metrics']
    print(f"  Fold {fr['fold']}: epoch={fr['best_epoch']} "
          f"Sens={m['sensitivity']:.3f} Spec={m['specificity']:.3f} "
          f"F1={m['f1']:.3f} AUC={m['auc_roc']:.3f} PRAUC={m['pr_auc']:.3f}")

## 5. Train ResNet-18 — 30 epochs + Early Stopping

Expected: ~15-25 min trên T4 GPU.

In [ ]:
!python3 m3_vision/scripts/train_vision_baseline.py \
  --project-root . \
  --encoder resnet18 \
  --epochs 30 \
  --folds 5 \
  --batch-size 16 \
  --lr 1e-4 \
  --weight-decay 1e-3 \
  --early-stop \
  --early-stop-patience 7 \
  --checkpoint-dir m3_vision/checkpoints/resnet18_30epoch \
  --output m3_vision/results/vision_resnet18_30epoch_metrics.json

In [ ]:
# Xem kết quả ResNet-18
import json
with open("m3_vision/results/vision_resnet18_30epoch_metrics.json") as f:
    result = json.load(f)

print("="*60)
print(f"ResNet-18 — {result['epochs']} epochs, {result['folds']} folds")
print("="*60)
for k, v in result['summary'].items():
    print(f"  {k:15s}: {v['mean']:.4f} ± {v['std']:.4f}")
print()
print("Per-fold:")
for fr in result['fold_results']:
    m = fr['best_metrics']
    print(f"  Fold {fr['fold']}: epoch={fr['best_epoch']} "
          f"Sens={m['sensitivity']:.3f} Spec={m['specificity']:.3f} "
          f"F1={m['f1']:.3f} AUC={m['auc_roc']:.3f} PRAUC={m['pr_auc']:.3f}")

## 6. Ablation — Attention Pooling vs Mean Pooling

So sánh 2 chiến lược pooling cho CCA encoder.

In [ ]:
!python3 m3_vision/scripts/ablation_pooling.py \
  --project-root . \
  --encoder custom_cnn \
  --epochs 20 \
  --folds 5 \
  --batch-size 16 \
  --early-stop \
  --early-stop-patience 5 \
  --output m3_vision/results/ablation_pooling_metrics.json

In [ ]:
# Xem kết quả ablation
import json
with open("m3_vision/results/ablation_pooling_metrics.json") as f:
    abl = json.load(f)

for pool_name in ["attention", "mean"]:
    r = abl[pool_name]
    print(f"\n{'='*50}")
    print(f"Pooling: {pool_name}")
    print(f"{'='*50}")
    for k, v in r['summary'].items():
        print(f"  {k:15s}: {v['mean']:.4f} ± {v['std']:.4f}")

## 7. Tổng hợp kết quả

In [ ]:
import json, os

results_dir = "m3_vision/results"
files = [
    "vision_custom_cnn_30epoch_metrics.json",
    "vision_resnet18_30epoch_metrics.json",
    "ablation_pooling_metrics.json",
]

metric_keys = ["sensitivity", "specificity", "f1", "auc_roc", "pr_auc"]

print(f"{'Model':<35s} {'Sens':>8s} {'Spec':>8s} {'F1':>8s} {'AUC':>8s} {'PRAUC':>8s}")
print("-" * 75)

def print_row(name, summary):
    vals = []
    for k in metric_keys:
        if k in summary:
            vals.append(f"{summary[k]['mean']:.3f}±{summary[k]['std']:.2f}")
        else:
            vals.append("   -   ")
    print(f"{name:<35s} {' '.join(vals)}")

# Custom CNN
f1 = os.path.join(results_dir, "vision_custom_cnn_30epoch_metrics.json")
if os.path.exists(f1):
    r = json.load(open(f1))
    print_row(f"Custom CNN ({r['epochs']}ep)", r['summary'])
else:
    print("Custom CNN 30ep: chưa chạy")

# ResNet-18
f2 = os.path.join(results_dir, "vision_resnet18_30epoch_metrics.json")
if os.path.exists(f2):
    r = json.load(open(f2))
    print_row(f"ResNet-18 ({r['epochs']}ep)", r['summary'])
else:
    print("ResNet-18 30ep: chưa chạy")

# Ablation
f3 = os.path.join(results_dir, "ablation_pooling_metrics.json")
if os.path.exists(f3):
    abl = json.load(open(f3))
    for pool_name in ["attention", "mean"]:
        if pool_name in abl:
            r = abl[pool_name]
            print_row(f"Custom CNN + {pool_name} pool", r['summary'])
else:
    print("Ablation: chưa chạy")

# Baseline cũ để so sánh
f4 = os.path.join(results_dir, "vision_custom_cnn_5epoch_metrics.json")
if os.path.exists(f4):
    r = json.load(open(f4))
    print_row(f"Custom CNN ({r['epochs']}ep, cũ)", r['summary'])

## 8. Tải kết quả về máy

Chọn 1 trong 2 cách:

In [ ]:
# --- Cách 1: Copy vào Google Drive ---
# import shutil
# dst = "/content/drive/MyDrive/M3_Results"
# os.makedirs(dst, exist_ok=True)
# for f in ["vision_custom_cnn_30epoch_metrics.json",
#           "vision_resnet18_30epoch_metrics.json",
#           "ablation_pooling_metrics.json"]:
#     src = os.path.join(results_dir, f)
#     if os.path.exists(src):
#         shutil.copy2(src, dst)
#         print(f"Copied {f} -> {dst}")
# # Copy checkpoints
# for ckpt_dir in ["custom_cnn_30epoch", "resnet18_30epoch"]:
#     src_dir = f"m3_vision/checkpoints/{ckpt_dir}"
#     if os.path.exists(src_dir):
#         shutil.copytree(src_dir, os.path.join(dst, ckpt_dir), dirs_exist_ok=True)
#         print(f"Copied checkpoints/{ckpt_dir}")

# --- Cách 2: Download trực tiếp ---
from google.colab import files
import glob

# Download JSON results
for pattern in ["m3_vision/results/*30epoch*.json", "m3_vision/results/ablation*.json"]:
    for f in glob.glob(pattern):
        print(f"Downloading {f}...")
        files.download(f)

# Download checkpoints (có thể lớn)
for pattern in ["m3_vision/checkpoints/*30epoch*/*.pt"]:
    for f in glob.glob(pattern):
        print(f"Downloading {f}...")
        files.download(f)